# Dataset Unification and Metadata Creation

This notebook prepares and standardizes the different ear datasets (AMI, BIPLab, UERC, and EICZA) into a unified structure and metadata format.

In [ ]:
import os
import shutil
import pandas as pd

# Ruta original
source_dir = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_orejas2026/AMI"
# Ruta destino
dest_dir = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/images"

os.makedirs(dest_dir, exist_ok=True)

metadata = []

for file in os.listdir(source_dir):
    if file.endswith(".jpg"):
        parts = file.split("_")
        subject = parts[0]          # 000
        pose = parts[1]             # back

        new_name = f"AMI_{subject}_{pose}.jpg"

        shutil.copy(
            os.path.join(source_dir, file),
            os.path.join(dest_dir, new_name)
        )

        metadata.append({
            "image_id": new_name.replace(".jpg",""),
            "subject_id": f"AMI_{subject}",
            "dataset": "AMI",
            "age_group": "adult",
            "image_path": f"images/{new_name}",
            "pose": pose
        })

df = pd.DataFrame(metadata)
df.to_csv("metadata_AMI.csv", index=False)


In [ ]:
import os
import re
import shutil
import pandas as pd

# ========= CONFIGURA ESTO =========
SOURCE_DIR = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_orejas2026/BIPLab/Ear"
DEST_IMAGES_DIR = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/images"
OUT_CSV = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/metadata_BIPLab.csv"
# ==================================

os.makedirs(DEST_IMAGES_DIR, exist_ok=True)

# Patrón esperado: ID001_SX_SAMPLE004.bmp
pattern = re.compile(r"^(ID\d+)_([A-Z]{2})_SAMPLE(\d+)\.(bmp|png|jpg|jpeg)$", re.IGNORECASE)

rows = []
skipped = []

for fname in os.listdir(SOURCE_DIR):
    fpath = os.path.join(SOURCE_DIR, fname)

    if not os.path.isfile(fpath):
        continue

    m = pattern.match(fname)
    if not m:
        skipped.append(fname)
        continue

    subj_raw = m.group(1).upper()      # ID001
    side_code = m.group(2).upper()     # SX / DX (u otro)
    sample_id = m.group(3).zfill(3)    # 004 -> 004
    ext = m.group(4).lower()

    # Mapear lado (si tu dataset usa SX/DX como izquierda/derecha)
    # Si no estás 100% seguro, deja unknown y ya lo ajustamos luego.
    ear_side = "left" if side_code == "SX" else ("right" if side_code == "DX" else "unknown")

    # Nombre nuevo (único global)
    # Ej: BIPLab_ID001_004.bmp
    new_name = f"BIPLab_{subj_raw}_{sample_id}.{ext}"

    # Copiar a carpeta unificada
    shutil.copy2(fpath, os.path.join(DEST_IMAGES_DIR, new_name))

    rows.append({
        "image_id": new_name.rsplit(".", 1)[0],          # sin extensión
        "subject_id": f"BIPLab_{subj_raw}",              # único global
        "dataset": "BIPLab",
        "age_group": "adult",
        "image_path": f"images/{new_name}",
        "ear_side": ear_side,
        "sample_id": int(sample_id),                     # útil para trazabilidad
        "original_filename": fname,
        # Campos de calidad (los dejas vacíos para rellenar luego si quieres)
        "quality_code": "",
        "q_blur": "",
        "q_partial": "",
        "q_crop": "",
        "q_occlusion": ""
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print(f"✅ Copiadas y renombradas: {len(df)} imágenes")
print(f"📄 Metadata guardado en: {OUT_CSV}")

if skipped:
    print(f"⚠️ Archivos saltados (no coinciden con patrón): {len(skipped)}")
    print("Ejemplos:", skipped[:10])


✅ Copiadas y renombradas: 300 imágenes
📄 Metadata guardado en: /Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/metadata_BIPLab.csv
⚠️ Archivos saltados (no coinciden con patrón): 6
Ejemplos: ['ear_features_matching.m', 'LDPEARS.m', 'lbpCodesExtraction.m', 'hog_150_250.mat', 'lbp_150_250_n8_r2.mat', 'lbp_150_250_n8_r1.mat']


In [ ]:
import os
import shutil
import pandas as pd

# ========= CONFIGURA ESTO =========
SOURCE_DIR = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_orejas2026/UERC/Dataset/Train Dataset"
DEST_IMAGES_DIR = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/images"
OUT_CSV = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/metadata_UERC_train.csv"
# ==================================

os.makedirs(DEST_IMAGES_DIR, exist_ok=True)

rows = []
skipped = []

def iter_images(subject_dir: str):
    """Devuelve lista de ficheros imagen dentro de un directorio (no recursivo)."""
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    for f in os.listdir(subject_dir):
        if f.lower().endswith(exts) and os.path.isfile(os.path.join(subject_dir, f)):
            yield f

# Recorrer carpetas de sujetos (0001, 0002, etc.)
subject_folders = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
])

for subj in subject_folders:
    subj_path = os.path.join(SOURCE_DIR, subj)

    # Lista estable para asignar índices 01, 02, 03...
    img_files = sorted(list(iter_images(subj_path)))

    if len(img_files) == 0:
        skipped.append((subj, "no_images"))
        continue

    for idx, fname in enumerate(img_files, start=1):
        src = os.path.join(subj_path, fname)
        ext = os.path.splitext(fname)[1].lower()  # incluye el punto

        # Nombre nuevo: UERC_<subject>_<idx>.ext
        # Ej: UERC_0001_01.jpg
        new_name = f"UERC_{subj}_{idx:02d}{ext}"

        dst = os.path.join(DEST_IMAGES_DIR, new_name)
        shutil.copy2(src, dst)

        rows.append({
            "image_id": new_name.rsplit(".", 1)[0],
            "subject_id": f"UERC_{subj}",
            "dataset": "UERC",
            "age_group": "adult",
            "image_path": f"images/{new_name}",
            # Campos opcionales
            "ear_side": "unknown",          # en UERC hay left/right, pero no viene explícito en nombres
            "original_filename": fname,
            "original_relpath": f"Train Dataset/{subj}/{fname}",
            # Calidad (para rellenar luego)
            "quality_code": "",
            "q_blur": "",
            "q_partial": "",
            "q_crop": "",
            "q_occlusion": ""
        })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print(f"✅ UERC Train: copiadas y renombradas {len(df)} imágenes")
print(f"📄 Metadata guardado en: {OUT_CSV}")

if skipped:
    print(f"⚠️ Carpetas saltadas: {len(skipped)}")
    print("Ejemplos:", skipped[:10])


✅ UERC Train: copiadas y renombradas 2304 imágenes
📄 Metadata guardado en: /Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/metadata_UERC_train.csv


In [ ]:
import os
import re
import shutil
import pandas as pd

SOURCE_DIR = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_orejas2026/dataset_EICZA/jpgs"
DEST_IMAGES_DIR = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/images"
OUT_CSV = "/Users/rafasuarzz/Desktop/ULPGC/GCID/Cuarto/TFG/TFG_dataset_unificado/metadata_EICZA.csv"

os.makedirs(DEST_IMAGES_DIR, exist_ok=True)

# Patrón flexible:
# small_<SUBJECT>_<AGE><UNIT>_<Left/Right>_<True/False>_Cropped.ext
# SUBJECT puede incluir espacios y "copy"
pattern = re.compile(
    r"^small_(.+?)_([0-9]+)([MWD])_(Left|Right)_(True|False)_Cropped\.(jpg|jpeg|png)$",
    re.IGNORECASE
)

rows = []
skipped = []

def normalize_subject(raw: str) -> str:
    # Quita sufijo " copy" y espacios extra
    s = raw.strip()
    s = re.sub(r"\s+copy$", "", s, flags=re.IGNORECASE)  # elimina " copy" al final
    s = s.replace(" ", "")  # por si quedan espacios en medio
    return s

def age_to_days(value: int, unit: str) -> float:
    unit = unit.upper()
    if unit == "D":
        return float(value)
    if unit == "W":
        return float(value) * 7.0
    if unit == "M":
        # 1 mes ≈ 30.437 días (promedio)
        return float(value) * 30.437
    raise ValueError("Unidad desconocida")

for fname in sorted(os.listdir(SOURCE_DIR)):
    src_path = os.path.join(SOURCE_DIR, fname)
    if not os.path.isfile(src_path):
        continue

    m = pattern.match(fname)
    if not m:
        skipped.append(fname)
        continue

    subj_raw = m.group(1)                 # puede incluir " copy"
    age_value = int(m.group(2))           # número
    age_unit = m.group(3).upper()         # M/W/D
    side_raw = m.group(4).lower()
    tf_raw = m.group(5).lower()
    ext = m.group(6).lower()

    subj = normalize_subject(subj_raw)

    ear_side = "left" if side_raw == "left" else "right"
    is_true = 1 if tf_raw == "true" else 0

    age_days = age_to_days(age_value, age_unit)
    age_months = round(age_days / 30.437, 2)

    side_code = "L" if ear_side == "left" else "R"
    tf_code = "T" if is_true == 1 else "F"

    # Nombre nuevo estable y único
    new_name = f"EICZA_{subj}_{age_value}{age_unit}_{side_code}_{tf_code}.{ext}"
    dst_path = os.path.join(DEST_IMAGES_DIR, new_name)

    # Copiar (sobrescribe si ya existe; si prefieres "no sobrescribir", dímelo y lo cambio)
    shutil.copy2(src_path, dst_path)

    rows.append({
        "image_id": new_name.rsplit(".", 1)[0],
        "subject_id": f"EICZA_{subj}",
        "dataset": "EICZA",
        "age_group": "child",
        "age_value": age_value,
        "age_unit": age_unit,
        "age_days": round(age_days, 2),
        "age_months": age_months,
        "ear_side": ear_side,
        "is_true_crop": is_true,
        "rotation_flag": "",  # marcar manualmente (0/1)
        "image_path": f"images/{new_name}",
        "original_filename": fname,
        "quality_code": ""
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print(f"✅ EICZA: copiadas {len(df)} imágenes")
print(f"⚠️ Saltadas: {len(skipped)}")
print("Ejemplos saltadas:", skipped[:15])


✅ EICZA: copiadas 3544 imágenes
⚠️ Saltadas: 0
Ejemplos saltadas: []


In [ ]:
import re
from pathlib import Path
import pandas as pd


# ----------------- utilidades -----------------
def norm_col(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def load_csv(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [norm_col(c) for c in df.columns]
    return df


def map_age_group(x):
    if pd.isna(x):
        return x
    s = str(x).strip().lower()
    mapping = {
        "adult": "adulto",
        "adulto": "adulto",
        "child": "niño",
        "infant": "niño",
        "kid": "niño",
        "nino": "niño",
        "niño": "niño",
    }
    return mapping.get(s, x)


def coalesce_cols(df: pd.DataFrame, candidates: list[str]):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def norm_ear_side(value):
    """
    Normaliza múltiples codificaciones a 'left'/'right'.
    Si no reconoce, devuelve NA.
    """
    if pd.isna(value):
        return pd.NA
    s = str(value).strip().lower()

    mapping = {
        # inglés
        "left": "left",
        "right": "right",
        "l": "left",
        "r": "right",
        # español
        "izquierda": "left",
        "derecha": "right",
        "izq": "left",
        "der": "right",
        # codificaciones típicas
        "sx": "left",   # sinistra
        "dx": "right",  # destra
    }
    if s in mapping:
        return mapping[s]

    # patrones dentro de strings (por si viene en nombres tipo "..._Left_..." etc.)
    if "left" in s or "_l" in s or " sx" in s or "_sx" in s:
        return "left"
    if "right" in s or "_r" in s or " dx" in s or "_dx" in s:
        return "right"

    return pd.NA


def norm_age_unit(u):
    if pd.isna(u):
        return pd.NA
    s = str(u).strip().lower()
    mapping = {
        "day": "days", "days": "days", "d": "days",
        "week": "weeks", "weeks": "weeks", "w": "weeks",
        "month": "months", "months": "months", "m": "months",
        "year": "years", "years": "years", "y": "years",
    }
    return mapping.get(s, s)


# ----------------- reglas ear_side por dataset -----------------
def infer_ear_side_ami(df: pd.DataFrame) -> pd.Series:
    """
    Regla que me diste:
      - pose == 'back' -> left
      - resto -> right
    Si no hay pose, devuelve NA.
    """
    if "pose" not in df.columns:
        return pd.Series([pd.NA] * len(df), index=df.index)
    pose = df["pose"].astype(str).str.strip().str.lower()
    return pose.apply(lambda p: "left" if p == "back" else "right")


def infer_ear_side_from_existing(df: pd.DataFrame) -> pd.Series:
    """
    Si existe alguna columna 'ear_side' o similar, la normaliza.
    """
    col = coalesce_cols(df, ["ear_side", "side", "ear", "laterality"])
    if col is None:
        return pd.Series([pd.NA] * len(df), index=df.index)
    return df[col].apply(norm_ear_side)


def infer_ear_side_uerc_heuristic(df: pd.DataFrame) -> pd.Series:
    """
    UERC: NO hay patrón fiable según comentas.
    Dejamos en NA salvo que venga ya indicado en alguna columna.
    Si quieres intentar heurística con original_filename/original_relpath/image_path,
    la activamos aquí (pero por defecto no inventamos).
    """
    s = infer_ear_side_from_existing(df)
    # Si quieres heurística, descomenta estas líneas:
    # if s.isna().all():
    #     col = coalesce_cols(df, ["original_filename", "original_relpath", "image_path", "image_id"])
    #     if col is not None:
    #         s = df[col].apply(norm_ear_side)
    return s


# ----------------- main -----------------
def build_unified_metadata(
    path_ami: str | Path,
    path_biplab: str | Path,
    path_eicza: str | Path,
    path_uerc: str | Path,
    output_path: str | Path = "metadata_unificado_final.csv",
) -> pd.DataFrame:
    ami = load_csv(path_ami)
    biplab = load_csv(path_biplab)
    eicza = load_csv(path_eicza)
    uerc = load_csv(path_uerc)

    datasets = [
        ("AMI", ami),
        ("BIPLab", biplab),
        ("EICZA", eicza),
        ("UERC", uerc),
    ]

    out_frames = []

    for name, df in datasets:
        out = pd.DataFrame()

        # base
        out["image_id"] = df["image_id"] if "image_id" in df.columns else pd.NA
        out["subject_id"] = df["subject_id"] if "subject_id" in df.columns else pd.NA
        out["dataset"] = df["dataset"] if "dataset" in df.columns else name
        out["age_group"] = df["age_group"].apply(map_age_group) if "age_group" in df.columns else pd.NA
        out["image_path"] = df["image_path"] if "image_path" in df.columns else pd.NA

        # ear_side por dataset
        if name == "AMI":
            out["ear_side"] = infer_ear_side_ami(df)
        elif name in ("BIPLab", "EICZA"):
            out["ear_side"] = infer_ear_side_from_existing(df)
        else:  # UERC
            out["ear_side"] = infer_ear_side_uerc_heuristic(df)

        # edad (EICZA value+unit, resto -1/none)
        if name == "EICZA":
            col_age_value = coalesce_cols(df, ["age_value", "age", "edad"])
            col_age_unit = coalesce_cols(df, ["age_unit", "unit", "age_units", "edad_unit"])

            out["age_value"] = pd.to_numeric(df[col_age_value], errors="coerce") if col_age_value else pd.NA
            out["age_unit"] = df[col_age_unit].apply(norm_age_unit) if col_age_unit else pd.NA
        else:
            out["age_value"] = -1
            out["age_unit"] = "none"

        # calidad: inicializar en blanco (aunque vengan valores)
        out["quality_code"] = ""
        out["q_blur"] = ""
        out["q_partial"] = ""
        out["q_crop"] = ""
        out["q_occlusion"] = ""

        out_frames.append(out)

    merged = pd.concat(out_frames, ignore_index=True)

    # columnas finales en el orden pedido
    final_cols = [
        "image_id",
        "subject_id",
        "dataset",
        "age_group",
        "image_path",
        "ear_side",
        "age_value",
        "age_unit",
        "quality_code",
        "q_blur",
        "q_partial",
        "q_crop",
        "q_occlusion",
    ]
    merged = merged[final_cols]

    merged.to_csv(output_path, index=False)
    return merged


if __name__ == "__main__":
    df = build_unified_metadata(
        path_ami="TFG_dataset_unificado/metadata_AMI.csv",
        path_biplab="TFG_dataset_unificado/metadata_BIPLab.csv",
        path_eicza="TFG_dataset_unificado/metadata_EICZA.csv",
        path_uerc="TFG_dataset_unificado/metadata_UERC_train.csv",
        output_path="TFG_dataset_unificado/metadata_unificado_final.csv",
    )

    print("✅ metadata unificado creado:", "metadata_unificado_final.csv")
    print("Filas/Columnas:", df.shape)
    print("\nEar_side (conteo):\n", df["ear_side"].value_counts(dropna=False))
    print("\nUERC ear_side NA (%):")
    uerc_mask = df["dataset"].astype(str).str.contains("UERC", na=False)
    if uerc_mask.any():
        print((df.loc[uerc_mask, "ear_side"].isna().mean() * 100), "%")


✅ metadata unificado creado: metadata_unificado_final.csv
Filas/Columnas: (6848, 13)

Ear_side (conteo):
 ear_side
right    2621
NaN      2304
left     1923
Name: count, dtype: int64

UERC ear_side NA (%):
100.0 %
